# UPN | Hands-on 02: Spark en modo local y operaciones con RDD
**Semana 12 · Unidad III · JupyterLab · Datos simulados**

**Caso resuelto:** análisis del consumo operativo en Agua, Energía y Residuos. Después, resuelva un desafío con datos ampliados.

**Versión 1.1:** diagnóstico de conexión Python/JVM y validaciones de secuencia de ejecución.

## Antes de comenzar
Coloque este notebook junto a `datos_consumo_24.csv` y `datos_consumo_100k.csv`. Active el kernel del entorno Conda que tenga PySpark. **No instale paquetes desde las celdas**; use Anaconda Prompt.

La ejecución `local[2]` utiliza recursos de una computadora: **no equivale a un clúster multinodo**.

**Importante:** tras reiniciar el kernel, vuelva a ejecutar desde el paso 1. El paso 1B es una prueba real de comunicación con el worker Python.

In [1]:
# 1. Comprobar el entorno y configurar PySpark ANTES de iniciar Spark
import os
import sys
import socket

print("Python:", sys.version.split()[0])
print("Ejecutable del kernel:", sys.executable)

# El worker debe utilizar el mismo intérprete que este kernel de JupyterLab.
os.environ["PYSPARK_PYTHON"] = sys.executable

# Para el laboratorio en UNA computadora: intentar la comunicación por loopback.
# No cambia el equipo a un clúster ni reemplaza las reglas del firewall.
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

try:
    import pyspark
except ImportError as exc:
    raise RuntimeError(
        "PySpark no está instalado en el kernel actual. Active el entorno Conda "
        "correcto (por ejemplo, etl_bi) y reinicie JupyterLab."
    ) from exc

print("PySpark:", pyspark.__version__)
print("Nombre del equipo:", socket.gethostname())

from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[2]")
         .appName("UPN-S12-RDD-v1.1")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("Spark:", spark.version)
print("Modo:", sc.master)
print("Python usado por el worker:", sc.pythonExec)


Python: 3.11.17
Ejecutable del kernel: C:\Users\UPN\.conda\envs\etl_bi\python.exe
PySpark: 4.2.0
Nombre del equipo: ULNLABB205PC31
Spark: 4.2.0
Modo: local[2]
Python usado por el worker: C:\Users\UPN\.conda\envs\etl_bi\python.exe


In [4]:
# 1B. Prueba de ejecución REAL de un worker Python (no solo creación del RDD)
# Si aquí falla, NO continúe con el caso: revise el entorno primero.
try:
    prueba = sc.parallelize([10, 20, 30, 40], 2)
    cantidad_prueba = prueba.count()
    suma_prueba = prueba.sum()
    assert cantidad_prueba == 4
    assert suma_prueba == 100
    print("PRUEBA CORRECTA: 4 registros; suma = 100")
except Exception as exc:
    texto = str(exc)
    if ("Python worker failed to connect back" in texto or
            "Timed out while waiting for the Python worker" in texto):
        raise RuntimeError(
            "Spark no pudo establecer la conexión local con el worker Python. "
            "1) Reinicie el kernel y ejecute desde la primera celda; "
            "2) revise que el kernel sea el entorno Conda correcto y que "
            "sys.executable coincida con el Python esperado; "
            "3) revise si el firewall bloquea java.exe/python.exe en localhost; "
            "4) confirme que Python, Java y PySpark sean compatibles. "
            "No modifique todavía el archivo CSV. El error original se conserva "
            "como causa para diagnóstico."
        ) from exc
    raise


PRUEBA CORRECTA: 4 registros; suma = 100


## Si aparece un error en JupyterLab

**`NameError: registros is not defined`**: falta ejecutar el paso 2, donde se lee `datos_consumo_24.csv`. Al reiniciar el kernel se pierden las variables anteriores; ejecute las celdas nuevamente desde arriba.

**`Python worker failed to connect back` / `SocketTimeoutException`**: Spark pudo iniciarse, pero una tarea que requería Python no logró conectarse con su proceso worker. La prueba **1B** identifica este problema antes del caso. Revise el entorno Conda y el kernel de JupyterLab, la compatibilidad de Python/Java/PySpark y posibles bloqueos locales de `java.exe` o `python.exe`. No desactive el firewall como medida general.

**Importante:** configure `PYSPARK_PYTHON` y `SPARK_LOCAL_IP` **antes** de crear `SparkSession`. Si cambió la configuración, use **Kernel → Restart Kernel** y ejecute las celdas en orden. Si persiste el fallo, copie el error completo y los valores mostrados en el paso 1. La configuración propuesta es una medida de diagnóstico, **no garantiza** resolver todos los problemas de conexión. La práctica sigue siendo en modo local, no en un clúster multinodo.


## Caso guiado completamente resuelto
El objetivo es obtener consumos por sector y verificar cada resultado. Primero se analizan **24 registros**, luego se repite el cálculo sobre **100 000 registros simulados**.

In [6]:
# 2. Lectura de CSV mediante herramientas estándar de Python
# La lectura local se usa para facilitar la introducción a RDD.
# No representa un patrón de ingesta distribuida para grandes volúmenes.

import csv
from pathlib import Path

def leer_registros(nombre_archivo):
    ruta = Path(nombre_archivo)
    if not ruta.exists():
        raise FileNotFoundError(f"No se encontró {ruta}. Coloque CSV y notebook en la misma carpeta.")
    with ruta.open(encoding="utf-8", newline="") as archivo:
        lector = csv.DictReader(archivo)
        columnas = set(lector.fieldnames or [])
        requeridas = {"id_registro", "fecha", "region", "sector", "consumo_unidades"}
        if not requeridas.issubset(columnas):
            raise ValueError(f"Faltan columnas: {sorted(requeridas-columnas)}")
        return [(fila["id_registro"], fila["fecha"], fila["region"],
                 fila["sector"], int(fila["consumo_unidades"])) for fila in lector]

registros = leer_registros("datos_consumo_24.csv")
print("Registros cargados:", len(registros))
print("Primeros tres:", registros[:3])
assert len(registros) == 24


Registros cargados: 24
Primeros tres: [('C000001', '2026-10-01', 'Lima', 'Agua', 10), ('C000002', '2026-10-02', 'La Libertad', 'Energia', 11), ('C000003', '2026-10-03', 'Arequipa', 'Residuos', 12)]


In [7]:
# 3. Crear un RDD con tres particiones
# La variable 'registros' se obtiene ejecutando el PASO 2.
if "registros" not in globals():
    raise RuntimeError(
        "No existe la variable 'registros'. Ejecute primero el PASO 2 "
        "(lectura de datos_consumo_24.csv). Tras reiniciar el kernel, "
        "vuelva a ejecutar las celdas en orden."
    )
if "sc" not in globals():
    raise RuntimeError("No existe SparkContext: ejecute primero el PASO 1.")
if len(registros) != 24:
    raise ValueError(f"Se esperaban 24 registros; se cargaron {len(registros)}.")

rdd = sc.parallelize(registros, 3)
print("Particiones:", rdd.getNumPartitions())
print("Registros:", rdd.count())
assert rdd.getNumPartitions() == 3
assert rdd.count() == 24


Particiones: 3
Registros: 24


In [8]:
# 4. Transformaciones y acciones
# filter y map son transformaciones; count y take son acciones.
agua = rdd.filter(lambda x: x[3] == "Agua")
consumos_agua = agua.map(lambda x: x[4])
print("Filas de Agua:", agua.count())
print("Primeros consumos de Agua:", consumos_agua.take(4))
assert agua.count() == 8


Filas de Agua: 8
Primeros consumos de Agua: [10, 13, 16, 19]


In [9]:
# 5. Agregar con reduceByKey, sin traer todas las filas al driver
por_sector = rdd.map(lambda x: (x[3], (x[4], 1)))
resumen = por_sector.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
)
# collect es aceptable AQUÍ porque solo devuelve tres grupos.
resultado = dict(resumen.collect())
for sector in sorted(resultado):
    suma, cantidad = resultado[sector]
    print(f"{sector}: registros={cantidad}, consumo_total={suma}, promedio={suma/cantidad:.2f}")
assert {k: v[1] for k, v in resultado.items()} == {"Agua": 8, "Energia": 8, "Residuos": 8}


Agua: registros=8, consumo_total=164, promedio=20.50
Energia: registros=8, consumo_total=172, promedio=21.50
Residuos: registros=8, consumo_total=180, promedio=22.50


**Interpretación:** las transformaciones describen un trabajo. Las acciones (por ejemplo `count`) provocan su evaluación. `reduceByKey` agrupa las claves y combina valores. El orden de los grupos no está garantizado; por eso se usa `sorted` al imprimir.

In [10]:
# 6. Repetir con 100 000 registros simulados
# NOTA: se leen primero en la memoria de Python y después se distribuyen
# localmente con parallelize. Para archivos grandes, usar una fuente distribuida.
registros_grandes = leer_registros("datos_consumo_100k.csv")
rdd_grande = sc.parallelize(registros_grandes, 4)
print("Filas:", rdd_grande.count())
print("Particiones:", rdd_grande.getNumPartitions())
assert rdd_grande.count() == 100000

conteo_sectores = dict(rdd_grande.map(lambda x: (x[3], 1))
                      .reduceByKey(lambda a, b: a + b).collect())
print("Registros por sector:", sorted(conteo_sectores.items()))
assert conteo_sectores == {"Agua": 33334, "Energia": 33333, "Residuos": 33333}


Filas: 100000
Particiones: 4
Registros por sector: [('Agua', 33334), ('Energia', 33333), ('Residuos', 33333)]


## Experimento con particiones
Mantenga el mismo cálculo y modifique únicamente la cantidad de particiones. Registre los resultados; no suponga que más particiones siempre será más rápido.

In [8]:
# 7. Experimento: ¿más particiones significa necesariamente menos tiempo?
from time import perf_counter

def medir_conteo(datos, particiones, repeticiones=3):
    valores = []
    for _ in range(repeticiones):
        inicio = perf_counter()
        trabajo = sc.parallelize(datos, particiones)
        # Hay transformación y acción; no se mide solo la creación del RDD.
        cantidad = trabajo.filter(lambda x: x[4] >= 30).count()
        valores.append(perf_counter() - inicio)
    return cantidad, valores

for p in (1, 2, 4):
    cantidad, duraciones = medir_conteo(registros_grandes, p)
    print(f"{p} particiones: {cantidad} filas; segundos={duraciones}")
# Los tiempos varían entre computadoras y ejecuciones. No existe un orden garantizado.


1 particiones: 51219 filas; segundos=[1.5426602999941679, 1.4343768000035197, 1.4986963999981526]
2 particiones: 51219 filas; segundos=[2.8135637000013958, 2.627799900001264, 2.639162400002533]
4 particiones: 51219 filas; segundos=[4.893684700000449, 5.006404899999325, 5.161058799996681]


## Desafío autónomo
Entregue código, resultados comprobados y explicación en lenguaje propio. El resultado es verificable; el tiempo de ejecución no es fijo.

In [ ]:
# DESAFÍO: completar sin consultar inicialmente la versión docente.
# A. Calcule la cantidad de registros por región con map y reduceByKey.
# B. Calcule la suma de consumos por sector con reduceByKey.
# C. Justifique por qué collect() es seguro sobre tres grupos,
#    pero no necesariamente sobre 100 000 registros.
# D. Explique qué cambió (o no) al usar 1, 2 y 4 particiones.

# Escriba aquí su solución.


### Preguntas de reflexión
1. ¿Qué prueba le permite afirmar que su agregación es correcta?
2. ¿Por qué la ejecución local no demuestra escalabilidad multinodo?
3. ¿Qué limitación tiene cargar un CSV completo en memoria de Python?
4. ¿Qué observaría en la interfaz Spark UI (`http://localhost:4040`, si está disponible)?

In [11]:
# 8. Cierre y limpieza del entorno (ejecutar al terminar todo el laboratorio)
# spark.stop() finaliza la sesión. Si vuelve a ejecutar el notebook,
# reinicie la sesión desde la primera celda.
spark.stop()
print("Sesión Spark detenida")


Sesión Spark detenida


**Fuentes oficiales:** [RDD Programming Guide](https://spark.apache.org/docs/latest/rdd-programming-guide) · [Instalación de PySpark](https://spark.apache.org/docs/latest/api/python/getting_started/install.html).